# Approach A — Baseline, Ceiling, and the Matched-Size Control

Reference points for every later approach.

| system | what it is |
|---|---|
| **baseline** | one classifier, all training rows |
| **ceiling** | one classifier per topic, routed by the true topic label |
| **matched** | one classifier per topic-sized random sample (controls for data volume) |

A naive ceiling mixes two effects: specialising helps, but each specialist sees ~1/7 the
data, which hurts. They roughly cancel, so ceiling − baseline alone looks like ≈0. The
matched control separates the two.

FineFake has 2,362 duplicated rows (over half of Reddit content); left in, 12.5% of the
test split leaked into training and inflated macro-F1 by ~2 points. `load_finefake()`
deduplicates by default — all numbers below are post-deduplication.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200)

from src import data, splits, diagnostics
from src.approaches import approach_a as A

## 1. Data

FineFake, normalised to `text`, `label`, `topic`, `platform`, `n_words`. `label`: 0 = fake, 1 = real.

In [2]:
df = data.load_finefake()
print(f"rows: {len(df):,}   labels: {df['label'].value_counts().to_dict()}")
pd.DataFrame({
    "n": df["topic"].value_counts(),
    "pct_fake": (df.groupby("topic")["label"].apply(lambda s: 100*(s==0).mean())).round(1),
})

rows: 15,474   labels: {0: 8155, 1: 7319}


,n,pct_fake
topic,,
Business,964,52.4
Conflict,1646,48.1
Entertainment,3214,53.3
Health,692,43.6
Politics,5252,49.0
Society,3596,61.3
Uncategorized,110,60.0


FineFake pools very different sources. Snopes entries are one-line claims;
AP News entries are full articles.

In [3]:
pd.DataFrame({
    "n": df["platform"].value_counts(),
    "median_words": df.groupby("platform")["n_words"].median().round(0),
    "pct_fake": (df.groupby("platform")["label"].apply(lambda s: 100*(s==0).mean())).round(1),
}).sort_values("n", ascending=False)

,n,median_words,pct_fake
platform,,,
snope,7556,18.0,67.3
reddit,2853,8.0,66.8
cnn,2252,268.0,10.4
washingtonpost,996,1020.0,32.1
twitter,875,21.0,62.3
apnews,707,883.0,9.2
cdc_gov,235,246.0,1.3


Short text skews fake, long text skews real — a model can score well on word
count alone (section 4).

## 2. Splits

Stratified on `(label, topic)`, committed to `artifacts/splits.json`, never regenerated.

In [4]:
sp = splits.load_splits()
splits.verify_splits(df, sp)        # raises if the data no longer matches
tr, va, te = splits.apply_splits(df, sp)
print({k: len(v) for k, v in sp.items()}, "— verified")

{'train': 10831, 'val': 2321, 'test': 2322} — verified


## 3. Run all three systems

In [5]:
res = A.run(df, sp, seed=42, write=False)
res[res["scope"] == "overall"][["system","n_train","n_test","macro_f1","roc_auc","mcc"]].round(4)

,system,n_train,n_test,macro_f1,roc_auc,mcc
0,baseline,10831,2322,0.7502,0.8308,0.5066
8,ceiling,10831,2306,0.7521,0.8251,0.5103


In [6]:
print(f"pooled ceiling - baseline = {A.gap(res):+.4f}   <- the naive number")

pooled ceiling - baseline = +0.0019   <- the naive number


Near zero — looks like specialising doesn't help, which is misleading.

### Decomposition

- `specialisation` = ceiling − matched (topic composition, data volume held equal)
- `starvation` = matched − baseline (cost of less data)
- `net` = ceiling − baseline = specialisation + starvation

In [7]:
dec = A.decompose(res)
dec

,n_train,baseline,matched,ceiling,specialisation,starvation,net
scope,,,,,,,
Politics,3676.0,0.7401,0.7417,0.7481,0.0064,0.0015,0.0080
Society,2517.0,0.7425,0.7201,0.7459,0.0258,-0.0224,0.0034
Entertainment,2250.0,0.8068,0.7765,0.8079,0.0314,-0.0302,0.0012
Conflict,1152.0,0.6467,0.6410,0.6437,0.0027,-0.0057,-0.0030
Business,675.0,0.7762,0.7813,0.7429,-0.0384,0.0050,-0.0334
Health,484.0,0.7958,0.7432,0.7864,0.0432,-0.0526,-0.0094


In [8]:
print(f"mean specialisation = {dec['specialisation'].mean():+.4f}"
      f"   positive in {(dec['specialisation'] > 0).sum()}/{len(dec)} topics")
print(f"mean starvation     = {dec['starvation'].mean():+.4f}")
print(f"mean net            = {dec['net'].mean():+.4f}")

mean specialisation = +0.0118   positive in 5/6 topics
mean starvation     = -0.0174
mean net            = -0.0055


Specialising helps in 5 of 6 topics (+0.012 avg); splitting the data costs
−0.017. They roughly cancel — well short of the ~3 points MDFEND/M³FEND report on
Weibo21, which shares parameters across domains instead of partitioning (Approach C).

## 4. The confound

How much of the score survives without reading the text at all?

In [9]:
base_f1 = float(res[(res.scope=="overall") & (res.system=="baseline")]["macro_f1"].iloc[0])
diagnostics.report(tr, te, baseline_f1=base_f1)

--- text-free shortcut baselines ---


                  probe  macro_f1  roc_auc
         majority class    0.3452      NaN
  log(word count) alone    0.7145   0.7771
platform identity alone    0.6985   0.7109

  real baseline macro_f1 = 0.7502
  best text-free probe   = 0.7145
  contribution of text   = +0.0358

--- length signal within each topic ---


        topic  n_train  length_only_f1  corr_len_real
     Business      675          0.7835         0.6403
     Conflict     1152          0.6373         0.3770
Entertainment     2250          0.7575         0.5402
       Health      484          0.7269         0.5805
     Politics     3676          0.6920         0.3941
      Society     2517          0.7151         0.4528
Uncategorized       77          0.5608         0.0574


`log(word count)` alone lands close to the full baseline — one feature, no
text. The strongest coefficients confirm it: a Reddit tag and stopwords, not
deception cues.

In [10]:
for term, weight in A.inspect_terms(res, 12):
    print(f"   {term:<22} {weight:+.3f}")

   haff                   +8.424
   psbattle               -6.582
   and                    +4.489
   vs                     +3.334
   in                     +2.928
   said                   +2.857
   as                     +2.551
   2021                   +2.366
   mr                     +2.326
   the                    +2.284
   after                  +2.123
   at                     +2.062


## 5. Topic or platform?

Topic and platform are tangled — Health skews to `cdc_gov` (1.1% fake), Entertainment
to `snope`/`reddit` (67–76% fake). A "topic specialist" could just be learning the
outlet's base rate. `scripts/test_topic_vs_platform.py` reruns the matched-size test
within each platform to check.

In [11]:
res_tp = pd.read_csv(REPO / "results" / "topic_vs_platform.csv")
res_tp

,platform,topic,n_train,n_test,pct_fake,specialist,matched,gain
0,snope,Business,382,88,79.1,0.4969,0.5339,-0.0370
1,snope,Conflict,734,156,62.1,0.5478,0.4884,0.0594
2,snope,Entertainment,937,216,61.5,0.6744,0.5836,0.0907
3,snope,Health,260,60,69.2,0.4563,0.4376,0.0187
4,snope,Politics,1676,372,66.3,0.5976,0.6073,-0.0097
5,snope,Society,1256,277,71.9,0.6222,0.5426,0.0796
6,reddit,Entertainment,753,155,71.8,0.8685,0.8801,-0.0116
7,reddit,Politics,568,103,54.6,0.8830,0.8543,0.0286
8,reddit,Society,631,111,71.2,0.8746,0.8297,0.0449
9,cnn,Entertainment,346,67,4.0,0.4962,0.4931,0.0031


In [12]:
print(res_tp.groupby("platform").agg(
    cells=("gain","size"), mean_gain=("gain","mean"),
    wins=("gain", lambda s: int((s>0).sum()))).round(4).to_string())
print(f"\nOVERALL within-platform topic gain = {res_tp['gain'].mean():+.4f}"
      f"   positive in {(res_tp['gain']>0).sum()}/{len(res_tp)} cells")
print(f"across platforms (section 3) it was            {dec['specialisation'].mean():+.4f}")

                cells  mean_gain  wins
platform                              
apnews              1     0.0000     0
cnn                 3    -0.0097     1
reddit              3     0.0206     2
snope               6     0.0336     4
twitter             2     0.0255     2
washingtonpost      1    -0.0164     0

OVERALL within-platform topic gain = +0.0168   positive in 9/16 cells
across platforms (section 3) it was            +0.0118


A positive effect survives (+0.017), but weak and inconsistent — 9/16 cells
gain. `cnn` and `apnews` are 12.9%/14.6% fake, lopsided enough that every model
collapses to "real" (macro-F1 0.46–0.50); their ≈0 gains say little. Where both
classes are present — snope, reddit, twitter — specialisation delivers 3–9 points.

## 6. What Approach A establishes

1. Specialists beat generalists, but only just: +0.012 across topics (5/6), +0.017
   within platform (9/16) — well below the ~3 points MDFEND reports on Weibo21.
2. Hard partitioning throws that gain away: net ≈ 0, since −0.017 starvation cancels it.
3. Two corrections matter before any number means anything: deduplication (done, ~2
   points of baseline) and the length/platform shortcut (`log(word count)` alone → 0.72).

### Carried forward

- Use `artifacts/splits.json` unchanged; never regenerate it.
- Reference points: baseline 0.7502, specialisation effect +0.012 — not the pooled
  ceiling, which is confounded.
- Always load with `deduplicate=True` (default).
- Approach C (mixture-of-experts) responds directly to finding 2.